# Azalyst Alpha Engine v2.0

**Architecture**: Walk-forward | 65 features | IC label | Purged CV | XGBoost GPU

**Strategy**: Train on Year 1+2 → Test on Year 3 (out-of-sample) → Quarterly retrain

**Features**: Returns | Volume | Volatility | Technical | Microstructure | WorldQuant 101 | Regime | Signal Processing

**Click Run All and wait for results.**

In [ ]:
# ── Cell 1: Imports & Paths ───────────────────────────────────────────────────
# Azalyst Alpha Engine v2.0
# Architecture: Walk-forward | 65 features | IC label | Purged CV | XGBoost GPU
import os, sys, time, json, gc, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings('ignore')

DATA_DIR    = '/kaggle/input/datasets/dhirajsuryavanshi/binance-data-5min-300-coins-3years'
WORK_DIR    = '/kaggle/working'
RESULTS_DIR = f'{WORK_DIR}/results'

for d in [RESULTS_DIR, f'{RESULTS_DIR}/models', f'{RESULTS_DIR}/shap']:
    os.makedirs(d, exist_ok=True)

parquet_files = sorted(Path(DATA_DIR).rglob('*.parquet'))
if not parquet_files:
    parquet_files = sorted(Path(DATA_DIR).parent.rglob('*.parquet'))
    if parquet_files:
        DATA_DIR = str(parquet_files[0].parent)

print(f'DATA_DIR    : {DATA_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'Parquet files found: {len(parquet_files)}')

In [ ]:
# ── Cell 2: GPU Setup ─────────────────────────────────────────────────────────
import subprocess, numpy as np, xgboost as xgb, lightgbm as lgb

try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    print(r.stdout.decode()[:300])
except: pass

HAS_GPU = False
USE_XGB_GPU = False

try:
    _x = np.random.rand(200, 5).astype(np.float32)
    _y = np.random.randint(0, 2, 200)
    xgb.XGBClassifier(device='cuda', n_estimators=3, verbosity=0).fit(_x, _y)
    HAS_GPU = True
    USE_XGB_GPU = True
    print(f'XGBoost {xgb.__version__}: CUDA CONFIRMED')
except Exception as e:
    print(f'XGBoost CUDA failed: {e} — using CPU')

import builtins
builtins.USE_XGB_GPU = USE_XGB_GPU
builtins.HAS_GPU = HAS_GPU

import psutil
def mem_gb():
    return psutil.virtual_memory().used / 1e9

def aggressive_gc():
    gc.collect()
    try:
        import cupy as cp
        cp.get_default_memory_pool().free_all_blocks()
    except: pass

print(f'\nHAS_GPU={HAS_GPU} | USE_XGB_GPU={USE_XGB_GPU}')
print(f'RAM available: {psutil.virtual_memory().available/1e9:.1f} GB')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
# Strategy parameters
STOP_LOSS_PCT   = -2.0       # tighter stop
TAKE_PROFIT_PCT =  5.0       # wider TP
TOP_QUANTILE    =  0.15      # top 15% signals (more selective)
FEE_RATE        =  0.001
ROUND_TRIP_FEE  =  FEE_RATE * 2

# Architecture
SCORE_RESAMPLE   = '4h'      # scoring timeframe
TRAIN_RESAMPLE   = '5min'    # training timeframe (native)
RETRAIN_WEEKS    = 13        # quarterly retrain (13 weeks)
MIN_TRAIN_ROWS   = 500       # min rows to trigger retrain

# Feature columns — 65 features across 8 categories
FEATURE_COLS = [
    # 1. Returns (7)
    'ret_1bar','ret_1h','ret_4h','ret_1d','ret_2d','ret_3d','ret_1w',
    # 2. Volume (6)
    'vol_ratio','vol_ret_1h','vol_ret_1d','obv_change','vpt_change','vol_momentum',
    # 3. Volatility (7)
    'rvol_1h','rvol_4h','rvol_1d','vol_ratio_1h_1d','atr_norm','parkinson_vol','garman_klass',
    # 4. Technical (10)
    'rsi_14','rsi_6','macd_hist','bb_pos','bb_width','stoch_k','stoch_d','cci_14','adx_14','dmi_diff',
    # 5. Microstructure (6)
    'vwap_dev','amihud','kyle_lambda','spread_proxy','body_ratio','candle_dir',
    # 6. Price structure (6)
    'wick_top','wick_bot','price_accel','skew_1d','kurt_1d','max_ret_4h',
    # 7. WorldQuant-inspired alphas (8)
    'wq_alpha001','wq_alpha012','wq_alpha031','wq_alpha098',
    'cs_momentum','cs_reversal','vol_adjusted_mom','trend_consistency',
    # 8. Regime features (5)
    'vol_regime','trend_strength','corr_btc_proxy','hurst_exp','fft_strength',
]

# Bar constants for 5min
BARS_PER_HOUR = 12
BARS_PER_DAY  = 288
HORIZON_BARS  = 48   # 4h forward

print(f'Feature count: {len(FEATURE_COLS)}')
print(f'Strategy: SL={STOP_LOSS_PCT}% | TP={TAKE_PROFIT_PCT}% | Top {TOP_QUANTILE*100:.0f}%')
print(f'Architecture: Train Y1+Y2 → Test Y3 | Quarterly retrain')

In [ ]:
# ── Cell 4: Feature Engineering v2 ───────────────────────────────────────────
# 65 features across 8 categories
# Sources: WorldQuant 101, ML4T, Qlib Alpha158, QuantResearch

def _rsi(s, n):
    d  = s.diff()
    g  = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    ls = (-d).clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + g / ls.replace(0, np.nan))

def _ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def _hurst(ts, max_lag=20):
    """Hurst exponent — H>0.5 trending, H<0.5 mean-reverting"""
    lags = range(2, max_lag)
    tau  = [np.std(np.subtract(ts[lag:], ts[:-lag])) for lag in lags]
    try:
        m = np.polyfit(np.log(lags), np.log(tau), 1)
        return float(m[0])
    except:
        return 0.5

def load_ohlcv(path):
    try:
        df = pd.read_parquet(path)
        df.columns = [c.lower() for c in df.columns]
        ts_col = next((c for c in df.columns if c in ('timestamp','time','open_time')), None)
        if ts_col:
            col = df[ts_col]
            if pd.api.types.is_integer_dtype(col):
                df.index = pd.to_datetime(col, unit='ms', utc=True)
            else:
                df.index = pd.to_datetime(col, utc=True)
            df = df.drop(columns=[ts_col])
        elif isinstance(df.index, pd.DatetimeIndex):
            df.index = (df.index.tz_localize('UTC') if df.index.tz is None
                        else df.index.tz_convert('UTC'))
        else:
            df.index = pd.to_datetime(df.index, utc=True)
        df = df.sort_index()
        for c in ['open','high','low','close','volume']:
            if c not in df.columns:
                return None
        return df[['open','high','low','close','volume']].apply(
            pd.to_numeric, errors='coerce').dropna()
    except:
        return None


def build_features(df, timeframe='5min'):
    """65 features across 8 categories — v2 full feature set"""
    bph, bpd = BARS_PER_HOUR, BARS_PER_DAY
    if timeframe != '5min':
        mins = {'1h':60,'4h':240,'1d':1440}.get(timeframe, 5)
        bph  = max(1, 60 // mins)
        bpd  = max(1, 1440 // mins)

    c = df['close'].astype(np.float32)
    o = df['open'].astype(np.float32)
    h = df['high'].astype(np.float32)
    l = df['low'].astype(np.float32)
    v = df['volume'].astype(np.float32)
    f = pd.DataFrame(index=df.index, dtype=np.float32)

    # ── 1. Returns ─────────────────────────────────────────────────────────
    lr = np.log(c / c.shift(1))
    f['ret_1bar'] = lr
    f['ret_1h']   = np.log(c / c.shift(bph))
    f['ret_4h']   = np.log(c / c.shift(bph * 4))
    f['ret_1d']   = np.log(c / c.shift(bpd))
    f['ret_2d']   = np.log(c / c.shift(bpd * 2))
    f['ret_3d']   = np.log(c / c.shift(bpd * 3))
    f['ret_1w']   = np.log(c / c.shift(bpd * 5))

    # ── 2. Volume ──────────────────────────────────────────────────────────
    av = v.rolling(bpd, min_periods=bph).mean()
    f['vol_ratio']   = v / av.replace(0, np.nan)
    f['vol_ret_1h']  = np.log(v / v.shift(bph).replace(0, np.nan))
    f['vol_ret_1d']  = np.log(v / v.shift(bpd).replace(0, np.nan))
    # OBV change (On-Balance Volume momentum)
    obv = (np.sign(lr) * v).cumsum()
    f['obv_change']  = obv.diff(bph) / (obv.abs().rolling(bpd, min_periods=bph).mean() + 1e-8)
    # Volume-Price Trend change
    vpt = (lr * v).cumsum()
    f['vpt_change']  = vpt.diff(bph)
    # Volume momentum
    f['vol_momentum'] = v.rolling(bph, min_periods=2).mean() / v.rolling(bpd, min_periods=bph).mean()

    # ── 3. Volatility ──────────────────────────────────────────────────────
    f['rvol_1h']  = lr.rolling(bph,     min_periods=max(2, bph//2)).std()
    f['rvol_4h']  = lr.rolling(bph * 4, min_periods=bph).std()
    f['rvol_1d']  = lr.rolling(bpd,     min_periods=bph).std()
    f['vol_ratio_1h_1d'] = f['rvol_1h'] / f['rvol_1d'].replace(0, np.nan)
    # ATR normalized
    tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
    atr14 = tr.ewm(span=14, adjust=False).mean()
    f['atr_norm']     = atr14 / c.replace(0, np.nan)
    # Parkinson volatility (high-low based — less noisy than close-close)
    f['parkinson_vol']  = np.sqrt(1/(4*np.log(2)) * np.log(h/l.replace(0,np.nan))**2
                           ).rolling(bpd, min_periods=bph).mean()
    # Garman-Klass volatility
    gk = (0.5 * np.log(h/l.replace(0,np.nan))**2
          - (2*np.log(2)-1) * np.log(c/o.replace(0,np.nan))**2)
    f['garman_klass'] = gk.rolling(bpd, min_periods=bph).mean()

    # ── 4. Technical indicators ────────────────────────────────────────────
    f['rsi_14'] = _rsi(c, 14) / 100.0
    f['rsi_6']  = _rsi(c,  6) / 100.0
    # MACD histogram
    macd_line   = _ema(c, 12) - _ema(c, 26)
    signal_line = _ema(macd_line, 9)
    f['macd_hist'] = (macd_line - signal_line) / c.replace(0, np.nan)
    # Bollinger Bands
    ma20  = c.rolling(20, min_periods=10).mean()
    std20 = c.rolling(20, min_periods=10).std(ddof=0)
    bw    = (4 * std20).replace(0, np.nan)
    f['bb_pos']   = ((c - (ma20 - 2*std20)) / bw).clip(0, 1)
    f['bb_width'] = bw / ma20.replace(0, np.nan)
    # Stochastic
    low14  = l.rolling(14, min_periods=7).min()
    high14 = h.rolling(14, min_periods=7).max()
    k = ((c - low14) / (high14 - low14).replace(0, np.nan) * 100).clip(0, 100)
    f['stoch_k'] = k / 100.0
    f['stoch_d'] = k.rolling(3).mean() / 100.0
    # CCI
    tp = (h + l + c) / 3
    tp_ma = tp.rolling(14, min_periods=7).mean()
    tp_mad = (tp - tp_ma).abs().rolling(14, min_periods=7).mean()
    f['cci_14'] = ((tp - tp_ma) / (0.015 * tp_mad.replace(0, np.nan))).clip(-3, 3) / 3
    # ADX + DMI
    plus_dm  = pd.Series(np.where((h.diff()>0) & (h.diff()>(-l.diff())), h.diff(), 0), index=df.index)
    minus_dm = pd.Series(np.where((-l.diff()>0) & (-l.diff()>h.diff()), -l.diff(), 0), index=df.index)
    atr_smooth = tr.ewm(span=14, adjust=False).mean()
    plus_di  = 100 * plus_dm.ewm(span=14, adjust=False).mean() / atr_smooth.replace(0, np.nan)
    minus_di = 100 * minus_dm.ewm(span=14, adjust=False).mean() / atr_smooth.replace(0, np.nan)
    dx = (100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
    f['adx_14']  = dx.ewm(span=14, adjust=False).mean() / 100.0
    f['dmi_diff'] = (plus_di - minus_di) / 100.0

    # ── 5. Microstructure ─────────────────────────────────────────────────
    vwap = ((tp * v).rolling(bpd, min_periods=bph).sum() /
             v.rolling(bpd, min_periods=bph).sum().replace(0, np.nan))
    f['vwap_dev']     = (c - vwap) / c.replace(0, np.nan)
    f['amihud']       = (lr.abs() / v.replace(0, np.nan)).rolling(bpd, min_periods=bph).mean()
    # Kyle lambda proxy (price impact)
    f['kyle_lambda']  = (lr.abs() / (v * c).replace(0, np.nan)).rolling(bpd, min_periods=bph).mean()
    # Spread proxy (high-low / close)
    f['spread_proxy'] = (h - l) / c.replace(0, np.nan)
    # Candle structure
    rng = (h - l).replace(0, np.nan)
    f['body_ratio'] = (c - o).abs() / rng
    f['candle_dir'] = np.sign(c - o)

    # ── 6. Price structure ─────────────────────────────────────────────────
    f['wick_top']   = (h - c.clip(lower=o)) / rng
    f['wick_bot']   = (c.clip(upper=o) - l) / rng
    m1 = c.pct_change(bph)
    f['price_accel'] = m1 - m1.shift(bph)
    f['skew_1d']    = lr.rolling(bpd, min_periods=max(4, bph)).skew()
    f['kurt_1d']    = lr.rolling(bpd, min_periods=max(4, bph)).kurt()
    f['max_ret_4h'] = lr.rolling(max(2, bph*4), min_periods=bph).max()

    # ── 7. WorldQuant-inspired alphas ─────────────────────────────────────
    # Alpha 001: sign(ret) * rvol — vol-adjusted momentum
    f['wq_alpha001'] = np.sign(f['ret_1d']) * f['rvol_1d']
    # Alpha 012: sign(delta volume) * (-delta price) — volume reversal
    f['wq_alpha012'] = np.sign(v.diff()) * (-lr)
    # Alpha 031: (-1 * rank(rank(rank(rolling_corr(rank, volume))))) proxy
    # Simplified: negative rank of rolling price-volume corr
    pv_corr = pd.Series(index=df.index, dtype=np.float32)
    _c_rank = c.rank()
    pv_corr = _c_rank.rolling(bpd, min_periods=bph).corr(v)
    f['wq_alpha031'] = -pv_corr
    # Alpha 098: momentum vs its own vol (Sharpe-like)
    ret_5d = np.log(c / c.shift(bpd))
    f['wq_alpha098'] = ret_5d / (f['rvol_1d'].replace(0, np.nan))
    # Cross-sectional (within-bar) features — placeholders, filled in cs_rank
    f['cs_momentum']     = f['ret_4h']           # will be cs-ranked
    f['cs_reversal']     = -f['ret_1d']           # will be cs-ranked
    f['vol_adjusted_mom'] = f['ret_4h'] / f['rvol_4h'].replace(0, np.nan)
    # Trend consistency (consecutive bars same direction)
    s = np.sign(lr)
    f['trend_consistency'] = s.rolling(min(48, bpd//2), min_periods=max(2,bph)).sum()

    # ── 8. Regime features ────────────────────────────────────────────────
    # Volatility regime (0=low, 0.5=mid, 1=high)
    vol_pct = f['rvol_1d'].rolling(bpd*5, min_periods=bpd).rank(pct=True)
    f['vol_regime'] = vol_pct
    # Trend strength (ADX-based)
    f['trend_strength'] = f['adx_14']
    # BTC correlation proxy (autocorrelation as regime proxy)
    f['corr_btc_proxy'] = lr.rolling(bpd, min_periods=bph).corr(lr.shift(1))
    # Hurst exponent (computed on rolling window)
    hurst_vals = pd.Series(index=df.index, dtype=np.float32)
    # Approximation: R/S statistic rolling
    rs_vals = []
    close_arr = c.values
    for i in range(len(close_arr)):
        if i < bpd:
            rs_vals.append(np.nan)
            continue
        seg = close_arr[i-bpd:i]
        mean_seg = np.mean(seg)
        deviations = np.cumsum(seg - mean_seg)
        r = np.max(deviations) - np.min(deviations)
        s_val = np.std(seg)
        rs_vals.append(r / s_val if s_val > 0 else np.nan)
    f['hurst_exp'] = pd.Series(rs_vals, index=df.index, dtype=np.float32)
    # FFT strength (dominant frequency power)
    fft_strength = pd.Series(index=df.index, dtype=np.float32)
    win = min(bpd, 288)
    for i in range(len(close_arr)):
        if i < win:
            fft_strength.iloc[i] = np.nan
            continue
        seg = close_arr[i-win:i]
        seg_ret = np.diff(np.log(seg + 1e-10))
        fft_mag = np.abs(np.fft.rfft(seg_ret))
        fft_strength.iloc[i] = float(np.max(fft_mag[1:]) / (np.mean(fft_mag[1:]) + 1e-10))
    f['fft_strength'] = fft_strength.astype(np.float32)

    return f.replace([np.inf, -np.inf], np.nan).shift(1).astype(np.float32)


print('Feature engineering v2 defined — 65 features across 8 categories')
print('Categories: Returns | Volume | Volatility | Technical | Microstructure | Price | WorldQuant | Regime')

In [ ]:
# ── Cell 5: Data Pipeline ─────────────────────────────────────────────────────

def build_panel(data_dir, date_from, date_to, resample='4h', verbose=True):
    """Load all symbols, compute features, resample. Memory-efficient v2."""
    _resample_to_mins = {'5min':5,'1h':60,'4h':240,'1d':1440}
    tf_mins = _resample_to_mins.get(resample, 240)
    horizon_bars = max(1, 240 // tf_mins)
    MIN_RAW_BARS = BARS_PER_DAY * 3
    files = sorted(Path(data_dir).glob('*.parquet'))

    def _process(f):
        try:
            df = load_ohlcv(f)
            if df is None or len(df) < MIN_RAW_BARS:
                return None
            df = df[(df.index >= date_from) & (df.index < date_to)]
            if len(df) < MIN_RAW_BARS:
                return None
            feat = build_features(df, timeframe='5min')
            feat['future_ret'] = np.log(df['close'].shift(-horizon_bars) / df['close']).astype(np.float32)
            feat['symbol'] = f.stem
            avail = [c for c in FEATURE_COLS if c in feat.columns]
            if resample != '5min':
                feat_rs = feat.resample(resample).last().dropna(subset=avail, how='all')
            else:
                feat_rs = feat.dropna(subset=avail, how='all')
            return feat_rs if len(feat_rs) >= 5 else None
        except:
            return None

    frames = []
    with ThreadPoolExecutor(max_workers=2) as ex:
        futures = {ex.submit(_process, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            result = fut.result()
            if result is not None:
                frames.append(result)
            if verbose and i % 50 == 0:
                print(f'  {i}/{len(files)} scanned, {len(frames)} valid | RAM={mem_gb():.1f}GB')

    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames).sort_index()
    if verbose:
        syms = result['symbol'].nunique() if 'symbol' in result.columns else '?'
        print(f'  Panel: {len(result):,} rows, {syms} symbols [{resample}]')
    return result


def add_labels(df):
    """Add IC-compatible labels: future_ret + binary alpha_label"""
    df = df.copy()
    # Cross-sectional median label
    median_ts = df.groupby(level=0)['future_ret'].transform('median')
    df['alpha_label'] = np.where(
        df['future_ret'].notna(),
        (df['future_ret'] > median_ts).astype(np.float32),
        np.nan
    )
    # Cross-sectional rank of future returns (for IC computation)
    df['future_ret_rank'] = df.groupby(level=0)['future_ret'].rank(pct=True)
    return df


def cs_rank(df, cols):
    """Cross-sectional percentile rank — vectorized"""
    avail = [c for c in cols if c in df.columns]
    ranked = df.copy()
    ranked[avail] = df[avail].groupby(level=0, sort=False).rank(pct=True, na_option='keep')
    return ranked


def compute_ic(y_pred, y_true):
    """Information Coefficient — rank correlation of predictions vs returns"""
    mask = ~(np.isnan(y_pred) | np.isnan(y_true))
    if mask.sum() < 10:
        return 0.0
    return float(stats.spearmanr(y_pred[mask], y_true[mask])[0])


print('Data pipeline v2 defined')
print('Features: 65 | Label: IC + binary | CS-rank + IC evaluation')

In [ ]:
# ── Cell 6: Date Range Discovery ──────────────────────────────────────────────
print('Scanning date range...')
starts, ends = [], []
files = sorted(Path(DATA_DIR).glob('*.parquet'))[:100]

for p in files:
    df = load_ohlcv(p)
    if df is not None and len(df) > 100:
        starts.append(df.index[0])
        ends.append(df.index[-1])

if not starts:
    raise RuntimeError('No valid files found')

global_start = min(starts)
global_end   = max(ends)
total_days   = (global_end - global_start).days
year_delta   = pd.Timedelta(days=total_days // 3)

year1_end = global_start + year_delta
year2_end = year1_end   + year_delta
year3_end = global_end

print(f'Global range: {global_start.date()} → {global_end.date()} ({total_days} days)')
print(f'Year 1 (train): {global_start.date()} → {year1_end.date()}')
print(f'Year 2 (train): {year1_end.date()} → {year2_end.date()}')
print(f'Year 3 (test) : {year2_end.date()} → {year3_end.date()}')

with open(f'{RESULTS_DIR}/date_config.json', 'w') as fh:
    json.dump({
        'global_start': global_start.isoformat(),
        'year1_end':    year1_end.isoformat(),
        'year2_end':    year2_end.isoformat(),
        'year3_end':    year3_end.isoformat(),
    }, fh, indent=2)

In [ ]:
# ── Cell 7: Model Training (XGBoost GPU + Purged CV) ─────────────────────────
import xgboost as xgb
from sklearn.preprocessing import RobustScaler

# Purged K-Fold for time series — prevents leakage
class PurgedTimeSeriesCV:
    def __init__(self, n_splits=5, gap=48):
        self.n_splits = n_splits
        self.gap = gap

    def split(self, X):
        n = len(X)
        fold_size = n // (self.n_splits + 1)
        for i in range(self.n_splits):
            train_end   = (i + 1) * fold_size
            val_start   = train_end + self.gap      # purge gap
            val_end     = val_start + fold_size
            if val_end > n:
                break
            train_idx = np.arange(0, train_end)
            val_idx   = np.arange(val_start, val_end)
            yield train_idx, val_idx


def make_xgb_model(use_gpu=True):
    params = dict(
        n_estimators    = 1000,
        learning_rate   = 0.02,
        max_depth       = 6,
        min_child_weight= 30,
        subsample       = 0.8,
        colsample_bytree= 0.7,
        colsample_bylevel=0.7,
        reg_alpha       = 0.1,
        reg_lambda      = 1.0,
        eval_metric     = 'auc',
        early_stopping_rounds = 50,
        verbosity       = 0,
        random_state    = 42,
    )
    if use_gpu:
        params['device'] = 'cuda'
    return xgb.XGBClassifier(**params)


def train_model(X, y, y_ret, feature_cols, label='', use_gpu=True):
    """Train with Purged CV — reports both AUC and IC"""
    scaler = RobustScaler()   # more robust than StandardScaler for fat-tailed finance data
    Xs = scaler.fit_transform(X)

    backend = 'XGBoost-GPU' if use_gpu else 'XGBoost-CPU'
    print(f'  Training [{label}]: {len(X):,} samples | {X.shape[1]} features | [{backend}]')

    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    aucs, ics = [], []

    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2:
            continue
        m = make_xgb_model(use_gpu)
        m.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)

        probs = m.predict_proba(Xs[val])[:, 1]
        try:
            auc = roc_auc_score(y[val], probs)
            aucs.append(auc)
        except: pass

        # IC on validation set
        if y_ret is not None:
            ic = compute_ic(probs, y_ret[val])
            ics.append(ic)

        print(f'    Fold {fold}: AUC={aucs[-1]:.4f}' +
              (f' | IC={ics[-1]:.4f}' if ics else ''))

    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    mean_ic  = float(np.mean(ics))  if ics  else 0.0
    icir     = float(np.mean(ics) / (np.std(ics) + 1e-8)) if ics else 0.0
    print(f'  CV Mean AUC={mean_auc:.4f} | IC={mean_ic:.4f} | ICIR={icir:.4f}')

    # Final model on full data
    final = make_xgb_model(use_gpu)
    split = int(len(Xs) * 0.9)
    final.fit(Xs[:split], y[:split], eval_set=[(Xs[split:], y[split:])], verbose=False)

    importance = pd.Series(
        final.feature_importances_,
        index=feature_cols, name='importance'
    ).sort_values(ascending=False)

    return final, scaler, importance, mean_auc, mean_ic, icir


def save_model(model, scaler, feature_cols, path, meta=None):
    payload = {'model': model, 'scaler': scaler, 'feature_cols': feature_cols}
    if meta: payload.update(meta)
    with open(path, 'wb') as fh:
        pickle.dump(payload, fh)


def load_model(path):
    with open(path, 'rb') as fh:
        obj = pickle.load(fh)
    return obj['model'], obj['scaler'], obj['feature_cols']


print('Model training v2 defined')
print('CV: Purged K-Fold (5 splits, gap=48 bars)')
print('Metrics: AUC + IC + ICIR')

In [ ]:
# ── Cell 8: Train on Year 1 + Year 2 ─────────────────────────────────────────
# Train on full 2-year history. Test on Year 3 only.
print('=' * 60)
print('  TRAINING: Year 1 + Year 2 (2 years combined)')
print('=' * 60)
t0 = time.time()
print(f'RAM start: {mem_gb():.1f} GB')

print('Loading Year 1 + Year 2 training data (5min native)...')
df_train = build_panel(DATA_DIR, global_start, year2_end,
                       resample='5min', verbose=True)

if df_train.empty:
    raise RuntimeError('No training data found')

print(f'Training panel: {len(df_train):,} rows | RAM={mem_gb():.1f}GB')

# Cast to float32 immediately
feat_cols_present = [c for c in FEATURE_COLS if c in df_train.columns]
df_train[feat_cols_present] = df_train[feat_cols_present].astype(np.float32)
aggressive_gc()

# Labels
df_train = add_labels(df_train)
avail    = [c for c in FEATURE_COLS if c in df_train.columns]
valid    = df_train.dropna(subset=avail + ['alpha_label'])

print(f'Valid labelled rows: {len(valid):,}')
print(f'Alpha rate: {valid["alpha_label"].mean()*100:.1f}%')

# Cross-sectional rank features
del df_train
aggressive_gc()
print(f'RAM before cs_rank: {mem_gb():.1f}GB')
valid = cs_rank(valid, avail)
print(f'RAM after cs_rank: {mem_gb():.1f}GB')

# Prepare arrays
X_train   = valid[avail].values.astype(np.float32)
y_train   = valid['alpha_label'].values.astype(int)
y_ret     = valid['future_ret'].values.astype(np.float32)

del valid
aggressive_gc()
print(f'RAM before training: {mem_gb():.1f}GB')

# Train
model_base, scaler_base, imp_base, auc, ic, icir = train_model(
    X_train, y_train, y_ret, avail,
    label='Y1+Y2_base', use_gpu=USE_XGB_GPU
)

# Save
base_model_path = f'{RESULTS_DIR}/models/model_base_y1y2.pkl'
save_model(model_base, scaler_base, avail, base_model_path,
           {'cv_auc': round(auc,4), 'ic': round(ic,4), 'icir': round(icir,4),
            'n_train': int(len(X_train))})
imp_base.to_csv(f'{RESULTS_DIR}/feature_importance_base.csv')

print(f'\nTop 15 features:')
for feat, val in imp_base.head(15).items():
    print(f'  {feat:<30}  {val:>8.1f}')

del X_train, y_train, y_ret
aggressive_gc()

print(f'\nTraining done in {(time.time()-t0)/60:.1f} min')
print(f'Base model AUC={auc:.4f} | IC={ic:.4f} | ICIR={icir:.4f}')
print(f'Model saved: {base_model_path}')
print(f'RAM: {mem_gb():.1f}GB')

In [ ]:
# ── Cell 9: Walk-Forward Test — Year 3 (Out-of-Sample) ────────────────────────
import builtins

# Make sure we have the base model from cell 8
try:
    model_base
    scaler_base
    avail
except NameError:
    print('Loading base model from disk...')
    model_base, scaler_base, avail = load_model(f'{RESULTS_DIR}/models/model_base_y1y2.pkl')

SCORE_RESAMPLE  = '4h'
SCORE_HORIZON   = 1    # 1 bar of 4h = 4h forward

def _simulate_trade(ohlcv_slice, signal, entry_price, max_bars, sl_p, tp_p):
    if len(ohlcv_slice) < 2:
        return entry_price, 'horizon'
    fut   = ohlcv_slice.iloc[1:max_bars+1]
    lows  = fut['low'].values
    highs = fut['high'].values
    if signal == 'BUY':
        sl_hit = np.where(lows  <= sl_p)[0]
        tp_hit = np.where(highs >= tp_p)[0]
    else:
        sl_hit = np.where(highs >= sl_p)[0]
        tp_hit = np.where(lows  <= tp_p)[0]
    sl_bar = sl_hit[0] if len(sl_hit) else max_bars + 1
    tp_bar = tp_hit[0] if len(tp_hit) else max_bars + 1
    if sl_bar < tp_bar and sl_bar <= max_bars: return sl_p,  'stop_loss'
    if tp_bar < sl_bar and tp_bar <= max_bars: return tp_p,  'take_profit'
    return float(fut.iloc[min(max_bars-1, len(fut)-1)]['close']), 'horizon'


def run_walkforward(model, scaler, feat_cols, year_start, year_end, year_label='Year3'):
    """Walk-forward test with quarterly retrain. No lookahead."""
    all_trades     = []
    weekly_summary = []
    weekly_returns = []
    ic_series      = []
    retrain_count  = 0
    current_model  = 'base_y1y2'

    week_starts = pd.date_range(
        year_start, year_end - pd.Timedelta(weeks=1),
        freq='W-MON', tz='UTC'
    )

    print(f'\n{"="*60}')
    print(f'  {year_label} Walk-Forward — {len(week_starts)} weeks')
    print(f'  {year_start.date()} → {year_end.date()}')
    print(f'{"="*60}')

    for wk_num, week_start in enumerate(week_starts, 1):
        week_end = week_start + pd.Timedelta(weeks=1)
        if week_end > year_end + pd.Timedelta(days=3):
            break

        t_wk = time.time()
        print(f'\n  Week {wk_num}/{len(week_starts)}: {week_start.date()} → {week_end.date()}')

        # Load week panel
        week_df = build_panel(DATA_DIR, week_start, week_end,
                              resample=SCORE_RESAMPLE, verbose=False)

        if week_df.empty:
            print('    [SKIP] No data')
            weekly_returns.append(0.0)
            weekly_summary.append({
                'week_num': wk_num, 'week_start': week_start.date().isoformat(),
                'week_end': week_end.date().isoformat(),
                'n_trades': 0, 'week_return_pct': 0.0, 'ic': 0.0,
                'retrained': False, 'model': current_model,
            })
            continue

        # Cross-sectional rank
        week_ranked = cs_rank(week_df, feat_cols)
        avail_wk    = [c for c in feat_cols if c in week_ranked.columns]

        # Batch scoring
        week_ranked = week_ranked.copy()
        week_ranked['prob']   = np.nan
        week_ranked['signal'] = 'HOLD'

        valid_mask = week_ranked[avail_wk].notna().all(axis=1)
        valid_rows = week_ranked[valid_mask]

        if len(valid_rows) >= 5:
            try:
                Xs_all    = scaler.transform(valid_rows[avail_wk].values.astype(np.float32))
                probs_all = model.predict_proba(Xs_all)[:, 1]

                prob_arr = week_ranked['prob'].values.copy()
                sig_arr  = week_ranked['signal'].values.copy()
                ts_index = week_ranked.index.get_level_values(0)
                valid_pos = np.where(valid_mask.values)[0]
                prob_arr[valid_pos] = probs_all

                for ts in pd.unique(ts_index[valid_pos]):
                    mask_ts = (ts_index == ts) & valid_mask.values
                    pos_ts  = np.where(mask_ts)[0]
                    if len(pos_ts) < 5: continue
                    p      = prob_arr[pos_ts]
                    n_long = max(1, int(len(pos_ts) * TOP_QUANTILE))
                    order  = np.argsort(p)
                    sig_arr[pos_ts[order[-n_long:]]] = 'BUY'
                    sig_arr[pos_ts[order[:n_long]]]  = 'SELL'

                week_ranked['prob']   = prob_arr
                week_ranked['signal'] = sig_arr

                # IC for this week
                fut_rets = week_ranked['future_ret'].values
                week_ic  = compute_ic(prob_arr, fut_rets)
                ic_series.append(week_ic)
                print(f'    IC={week_ic:.4f}')
            except Exception as e:
                print(f'    Scoring error: {e}')

        n_sigs = int((week_ranked['signal'] != 'HOLD').sum())
        print(f'    Signals: {n_sigs}')

        # Pre-load OHLCV for signal symbols
        sig_rows    = week_ranked[week_ranked['signal'].isin(['BUY','SELL'])]
        needed_syms = sig_rows['symbol'].dropna().unique() if 'symbol' in sig_rows.columns else []
        load_end    = week_end + pd.Timedelta(hours=12)
        ohlcv_cache = {}
        for sym in needed_syms:
            p = Path(DATA_DIR) / f'{sym}.parquet'
            if p.exists():
                raw = load_ohlcv(p)
                if raw is not None:
                    ohlcv_cache[sym] = raw[(raw.index >= week_start) & (raw.index <= load_end)]

        # Simulate trades
        max_bars    = SCORE_HORIZON * 12
        week_trades = []
        for ts, row in sig_rows.iterrows():
            sym    = str(row.get('symbol',''))
            signal = str(row['signal'])
            prob   = float(row.get('prob', 0.5))
            ohlcv  = ohlcv_cache.get(sym)
            if ohlcv is None: continue
            future = ohlcv[ohlcv.index > ts].head(max_bars + 10)
            if len(future) < 2: continue
            entry = float(future.iloc[0]['open'])
            if entry <= 0: continue
            if signal == 'BUY':
                sl_p = entry * (1 + STOP_LOSS_PCT/100)
                tp_p = entry * (1 + TAKE_PROFIT_PCT/100)
            else:
                sl_p = entry * (1 - STOP_LOSS_PCT/100)
                tp_p = entry * (1 - TAKE_PROFIT_PCT/100)
            exit_p, reason = _simulate_trade(future, signal, entry, max_bars, sl_p, tp_p)
            raw_ret = exit_p / entry - 1
            if signal == 'SELL': raw_ret = -raw_ret
            pnl = (raw_ret - ROUND_TRIP_FEE) * 100
            week_trades.append({
                'week_num': wk_num, 'signal_time': ts.isoformat(),
                'symbol': sym, 'signal': signal,
                'probability': round(prob,4),
                'entry': round(entry,8), 'exit': round(exit_p,8),
                'pnl_pct': round(pnl,4),
                'result': 'WIN' if pnl > 0 else 'LOSS',
                'exit_reason': reason,
            })

        all_trades.extend(week_trades)
        del ohlcv_cache
        gc.collect()

        # Weekly return
        if week_trades:
            pnls = np.array([t['pnl_pct'] for t in week_trades])
            wk_ret  = float(np.mean(pnls))
            win_rt  = float(np.mean(pnls > 0) * 100)
            wins    = pnls[pnls > 0]
            losses  = pnls[pnls < 0]
            pf      = float(np.sum(wins) / (-np.sum(losses))) if len(losses) > 0 and np.sum(losses) < 0 else 0.0
        else:
            wk_ret, win_rt, pf = 0.0, 0.0, 0.0

        weekly_returns.append(wk_ret / 100)
        print(f'    Return: {wk_ret:+.2f}% | WR: {win_rt:.0f}% | PF: {pf:.2f} | Trades: {len(week_trades)}')

        # Quarterly retrain — every 13 weeks
        retrained = False
        if wk_num % RETRAIN_WEEKS == 0:
            print(f'    QUARTERLY RETRAIN (week {wk_num})')
            retrain_count += 1
            train_end = week_end

            train_df = build_panel(DATA_DIR, global_start, train_end,
                                   resample='5min', verbose=False)
            if not train_df.empty:
                feat_p = [c for c in FEATURE_COLS if c in train_df.columns]
                train_df[feat_p] = train_df[feat_p].astype(np.float32)
                train_df = add_labels(train_df)
                valid2   = train_df.dropna(subset=feat_p + ['alpha_label'])
                del train_df
                aggressive_gc()

                if len(valid2) > MIN_TRAIN_ROWS:
                    valid2 = cs_rank(valid2, feat_p)
                    X2 = valid2[feat_p].values.astype(np.float32)
                    y2 = valid2['alpha_label'].values.astype(int)
                    y2_ret = valid2['future_ret'].values.astype(np.float32)
                    del valid2
                    aggressive_gc()

                    new_model, new_scaler, imp2, auc2, ic2, icir2 = train_model(
                        X2, y2, y2_ret, feat_p,
                        label=f'Y3_wk{wk_num}', use_gpu=USE_XGB_GPU)
                    del X2, y2, y2_ret
                    aggressive_gc()

                    model   = new_model
                    scaler  = new_scaler
                    feat_cols = feat_p
                    fname   = f'model_y3_week{wk_num:03d}.pkl'
                    current_model = fname
                    save_model(model, scaler, feat_cols,
                               f'{RESULTS_DIR}/models/{fname}',
                               {'cv_auc': round(auc2,4), 'ic': round(ic2,4)})
                    imp2.to_csv(f'{RESULTS_DIR}/feature_importance_y3_wk{wk_num}.csv')
                    print(f'    Retrained → AUC={auc2:.4f} | IC={ic2:.4f}')
                    retrained = True

        weekly_summary.append({
            'week_num': wk_num,
            'week_start': week_start.date().isoformat(),
            'week_end':   week_end.date().isoformat(),
            'n_trades':   len(week_trades),
            'week_return_pct': round(wk_ret, 4),
            'win_rate':   round(win_rt, 2),
            'profit_factor': round(pf, 4),
            'ic':  round(ic_series[-1] if ic_series else 0.0, 4),
            'retrained':  retrained,
            'model':      current_model,
        })

        del week_df, week_ranked
        gc.collect()
        print(f'    Week done in {time.time()-t_wk:.1f}s | Total retrains: {retrain_count}')

    # Save results
    wk_df = pd.DataFrame(weekly_summary)
    tr_df = pd.DataFrame(all_trades)
    wk_df.to_csv(f'{RESULTS_DIR}/weekly_summary_year3.csv', index=False)
    tr_df.to_csv(f'{RESULTS_DIR}/all_trades_year3.csv',     index=False)

    # IC statistics
    if ic_series:
        ic_arr = np.array(ic_series)
        print(f'\n  IC Stats: mean={np.mean(ic_arr):.4f} | std={np.std(ic_arr):.4f} | '
              f'ICIR={np.mean(ic_arr)/np.std(ic_arr):.4f} | '
              f'% positive={np.mean(ic_arr>0)*100:.0f}%')

    print(f'\n  {year_label} complete. Retrains: {retrain_count}')
    return wk_df, tr_df, ic_series


print('Walk-forward engine v2 defined')
print('Architecture: Quarterly retrain | IC tracking | No lookahead')

In [ ]:
# ── Cell 10: Run Year 3 Walk-Forward Test ─────────────────────────────────────
wk3, tr3, ic3 = run_walkforward(
    model_base, scaler_base, avail,
    year_start=year2_end, year_end=year3_end,
    year_label='Year3'
)

In [ ]:
# ── Cell 11: Performance Report ───────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def generate_report(wk_df, tr_df, ic_series, label='Year3'):
    if wk_df.empty or tr_df.empty:
        print('No results to report')
        return

    pnls    = tr_df['pnl_pct'].values
    rets    = wk_df['week_return_pct'].values / 100

    # Cumulative return
    cum_ret = (1 + pd.Series(rets)).cumprod() - 1

    # Annual return
    n_weeks  = len(rets)
    ann_ret  = float((1 + np.sum(rets)) ** (52/max(n_weeks,1)) - 1) * 100

    # Sharpe
    sharpe   = float(np.mean(rets) / (np.std(rets) + 1e-8) * np.sqrt(52))

    # Max drawdown
    cum_val  = (1 + pd.Series(rets)).cumprod()
    rolling_max = cum_val.cummax()
    drawdown = (cum_val - rolling_max) / rolling_max
    max_dd   = float(drawdown.min()) * 100

    # Trade stats
    wins     = pnls[pnls > 0]
    losses   = pnls[pnls < 0]
    win_rate = float(np.mean(pnls > 0) * 100)
    avg_win  = float(np.mean(wins))  if len(wins)  else 0
    avg_loss = float(np.mean(losses)) if len(losses) else 0
    pf       = float(np.sum(wins) / (-np.sum(losses))) if len(losses) and np.sum(losses)<0 else 0

    # IC stats
    if ic_series:
        ic_arr   = np.array(ic_series)
        mean_ic  = float(np.mean(ic_arr))
        icir     = float(np.mean(ic_arr) / (np.std(ic_arr) + 1e-8))
        ic_pos   = float(np.mean(ic_arr > 0) * 100)
    else:
        mean_ic = icir = ic_pos = 0.0

    print('\n' + '='*60)
    print(f'  AZALYST v2 — {label} OUT-OF-SAMPLE RESULTS')
    print('='*60)
    print(f'  Annualised Return : {ann_ret:+.1f}%')
    print(f'  Sharpe Ratio      : {sharpe:.2f}')
    print(f'  Max Drawdown      : {max_dd:.1f}%')
    print(f'  Total Trades      : {len(pnls):,}')
    print(f'  Win Rate          : {win_rate:.1f}%')
    print(f'  Avg Win           : {avg_win:+.2f}%')
    print(f'  Avg Loss          : {avg_loss:+.2f}%')
    print(f'  Profit Factor     : {pf:.2f}')
    print(f'  Mean IC           : {mean_ic:.4f}')
    print(f'  ICIR              : {icir:.4f}')
    print(f'  IC % Positive     : {ic_pos:.0f}%')
    print('='*60)

    summary = {
        'label': label,
        'annualised_return_pct': round(ann_ret, 2),
        'sharpe_ratio':          round(sharpe, 4),
        'max_drawdown_pct':      round(max_dd, 2),
        'total_trades':          int(len(pnls)),
        'win_rate_pct':          round(win_rate, 2),
        'avg_win_pct':           round(avg_win, 4),
        'avg_loss_pct':          round(avg_loss, 4),
        'profit_factor':         round(pf, 4),
        'mean_ic':               round(mean_ic, 4),
        'icir':                  round(icir, 4),
        'ic_pct_positive':       round(ic_pos, 1),
    }
    with open(f'{RESULTS_DIR}/performance_{label.lower()}.json', 'w') as fh:
        json.dump(summary, fh, indent=2)
    print(f'  Report saved → {RESULTS_DIR}/performance_{label.lower()}.json')

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f'Azalyst v2 — {label}', fontsize=14, fontweight='bold')

    # Cumulative return
    axes[0,0].plot(cum_ret.values * 100, color='steelblue', linewidth=1.5)
    axes[0,0].fill_between(range(len(cum_ret)), cum_ret.values * 100, 0,
                            alpha=0.2, color='steelblue')
    axes[0,0].axhline(0, color='black', linewidth=0.5)
    axes[0,0].set_title(f'Cumulative Return ({ann_ret:+.1f}% ann.)')
    axes[0,0].set_ylabel('Return %')

    # Weekly returns histogram
    axes[0,1].hist(rets * 100, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
    axes[0,1].axvline(0, color='red', linewidth=1)
    axes[0,1].set_title(f'Weekly Returns (Sharpe={sharpe:.2f})')
    axes[0,1].set_xlabel('Return %')

    # IC time series
    if ic_series:
        axes[1,0].bar(range(len(ic_series)), ic_series,
                      color=['steelblue' if x > 0 else 'salmon' for x in ic_series],
                      alpha=0.8)
        axes[1,0].axhline(0, color='black', linewidth=0.5)
        axes[1,0].axhline(mean_ic, color='green', linewidth=1.5, linestyle='--',
                           label=f'Mean IC={mean_ic:.4f}')
        axes[1,0].set_title(f'Weekly IC (ICIR={icir:.2f})')
        axes[1,0].legend(fontsize=8)

    # Trade PnL distribution
    axes[1,1].hist(pnls, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    axes[1,1].axvline(0, color='red', linewidth=1)
    axes[1,1].set_title(f'Trade P&L (WR={win_rate:.0f}%, PF={pf:.2f})')
    axes[1,1].set_xlabel('PnL %')

    plt.tight_layout()
    plot_path = f'{RESULTS_DIR}/performance_{label.lower()}.png'
    plt.savefig(plot_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'  Chart saved → {plot_path}')


generate_report(wk3, tr3, ic3, label='Year3')

In [ ]:
# ── Cell 12: Package Results ───────────────────────────────────────────────────
import shutil

zip_base = f'{WORK_DIR}/azalyst_v2_results'
shutil.make_archive(zip_base, 'zip', RESULTS_DIR)
zip_path = zip_base + '.zip'

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Results packaged: {zip_path} ({size_mb:.1f} MB)')
print('\nFiles included:')
for f in sorted(Path(RESULTS_DIR).rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(RESULTS_DIR)}  ({f.stat().st_size/1e3:.0f} KB)')
print('\nDownload from Kaggle Output tab → azalyst_v2_results.zip')